# Reading of Files in Spark from File-Based Source System for Both Batch and Stream
* Topic: Reading of Files in Spark from File-Based Source System for Both Batch and Stream
* Author: Oindrila Chakraborty

# What is Spark Partition?
* A <b>Spark Partition</b> refers to a <b>Chunk</b> of data that is actively being processed by one <b>Task</b> in an <b>Executor</b> in the <b>Cluster</b>, and, during the <b>Write Stage</b> gets mapped as a discrete file on <b>Disk</b> in the <b>Target Cloud Storage</b>.
* During execution, the number of <b>Spark Partitions</b> dictates the exact number of <b>Concurrent Tasks</b>, and, directly determines how many <b>Physical</b> files are created on <b>Disk</b> in the <b>Target Cloud Storage</b>.

# How Spark Configuration Option "spark.sql.files.maxPartitionBytes" Helps to Determines the Number of Spark Partitions to Create in Initial Batch Read Stage?
* The <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> signifies the maximum number of <b>Bytes</b> to pack into a single <b>Spark Partition</b> when reading data from only <b>File-Based Sources</b>, like - <b>Parquet</b>, <b>ORC</b>, <b>JSON</b>, and, <b>CSV</b> at the initial <b>Read Stage</b>.
* This configuration option determines the maximum size a chunk of data that can have to divide up the data files, which <b>Spark</b> reads during the initial <b>Read Stage</b> using, e.g., <b>spark.read.parquet()</b>, or, <b>spark.read.json()</b> etc.
* This configuration option can only determine the number of <b>Concurrent Tasks</b> to be created during initial <b>Read Stage</b>.
* This configuration option does not govern -
    * The size, or, count of the output files written back to <b>Disk</b>.
    * The size of the <b>Shuffle Partitions</b>, which is controlled by another <b>Spark</b> configuration option <b>spark.sql.shuffle.partitions</b>, or, <b>AQE</b>
    * The final size, or, count of the output file after a <b>Shuffle</b> occurs.
* <b>Number of Spark Partitions Created When Reading Large Files</b>:
    * If the <b>File-Based Source Directory</b> contains a single large uncompressed file, say, a file sized <b>1 GB</b>, which is read, then <b>Spark</b> first looks at configuration option <b>spark.sql.files.maxPartitionBytes</b>, which defaults to <b>128 MB</b>, and, then <b>Spark</b> will split that large file into roughly <b>8</b> smaller <b>Spark Partitions</b>, as <b>1024 MB / 128 MB = 8</b>.
    * Each <b>Spark Partition</b> is processed by a single <b>Task</b>, meaning <b>8 Tasks</b> can process the entire <b>1 GB</b> uncompressed file concurrently.
* <b>Number of Spark Partitions Created When Reading Thousands of Small Files</b>:
    * If the <b>File-Based Source Directory</b> contains thousands of tiny files, say, a file sized <b>2 MB</b>, then creating a dedicated <b>Spark Partition</b> for each of the tiny files would trigger massive <b>Scheduling</b>, and, <b>Metadata Loading Overhead</b>, <b>Task Overhead</b> on <b>Driver Memory</b>.
    * Instead, <b>Spark</b> uses this configuration option <b>spark.sql.files.maxPartitionBytes</b> to pack multiple small files together into a single <b>Spark Partition</b> until the total size of that <b>Spark Partition</b> gets close to the <b>128 MB</b> threshold, which creates lesser number of <b>Tasks</b> than <b>Spark</b> would have created if it created one <b>Spark Partition</b> per tiny file.
* The logic to pack the maximum number of <b>Bytes</b> into a single <b>Spark Partition</b> is also influenced by the <b>Spark</b> configuration option <b>spark.sql.files.openCostInBytes</b>, which adds a virtual <b>cost penalty</b> to open new files to prevent over-packing.

# When to Decrease the Value of the Spark Configuration Option "spark.sql.files.maxPartitionBytes"?
* <b>To Increase Concurrent Tasks, i.e., Parallelism</b>:
    * If the working <b>Cluster</b> is a large one, where many of the <b>CPU Cores</b> remain idle during execution, then reducing the value of the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> forces <b>Spark</b> to pack newly reduced maximum number of <b>Bytes</b> into more number of <b>Spark Partitions</b> each, which will increase the number of <b>Concurrent Tasks</b> than before, while reading data from only <b>File-Based Sources</b>, like - <b>Parquet</b>, <b>ORC</b>, <b>JSON</b>, and, <b>CSV</b> at the initial <b>Read Stage</b>.
    * Now, that there are more <b>Concurrent Tasks</b> created, more <b>CPU Cores</b> will be used at once, which will increase the <b>Parallelism</b> in the large <b>Cluster</b>.
* <b>To Prevent Out Of Memory (OOM) Errors</b>:
    * If the data from the input files has highly <b>Nested Structures</b>, or, contains <b>Complex Fields</b>, then the data would require some complex transformations to be implemented before being written to a <b>Delta Table</b>, like - an <b>explode ()</b> function, then those records, present in the <b>Spark Partition</b> of the default size of <b>128 MB</b>, can expand massively in the <b>Executor Memory</b>, and, can consume significantly more space than the raw file size on disk.
    * In this case, the size of the <b>Spark Partition</b> should be reduced to a smaller number, e.g., <b>32 MB</b>, or, <b>64 MB</b> to split the input files into smaller chunks, so that, when the data managed by each <b>Task</b> gets expanded in the <b>Memory</b>, then the <b>Memory</b> required by individual <b>Task</b> do not cause <b>Out Of Memory</b> (<b>OOM</b>) Error.

# When to Increase the Value of the Spark Configuration Option "spark.sql.files.maxPartitionBytes"?
* If the working <b>Cluster</b> is a large one with thousands of <b>CPU Cores</b>, and, using that <b>Cluster</b>, files of <b>Multi-Terabyte</b> size are read by <b>Spark</b> from <b>File-Based Source</b>, then millions of <b>Spark Partitions</b> of default size of <b>128 MB</b> will be created, and, assigned to each of the tiny <b>Tasks</b> for processing.
<br>Hence, millions of tiny <b>Tasks</b> will get created.
<br>In this case, the <b>Cluster</b> will waste more time in <b>Scheduling</b> the <b>Tasks</b>, rather than actually executing the code.
* Now, if the value of the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> is increased to <b>256 MB</b>, or, <b>512 MB</b>, then <b>Spark</b> creates fewer, but, larger <b>Spark Partitions</b>, and, each of these <b>Spark Partitions</b> get assigned to each <b>Tasks</b>.
<br>So, the number of <b>Tasks</b> are decreased significantly, which reduces the <b>Scheduling</b>, <b>Metadata Tracking Overhead</b> on the <b>Driver</b>, and, optimizes <b>Cluster Resource Throughput</b>.

# How Spark Configuration Option "spark.sql.files.openCostInBytes" Helps to Determines the Number of Spark Partitions to Create in Initial Read Stage?
* The <b>Spark</b> configuration option <b>spark.sql.files.openCostInBytes</b> estimates the the <b>cost</b> of opening a single file when <b>Spark</b> needs to group multiple input files into a single <b>Spark Partition</b> while reading data from only <b>File-Based Sources</b>, like - <b>Parquet</b>, <b>ORC</b>, <b>JSON</b>, and, <b>CSV</b> at the initial <b>Read Stage</b>.
* When <b>Spark</b> evaluates the input file size on <b>Disk</b> to decide how many <b>Concurrent Tasks</b> to create, <b>Spark</b> does not just look at the actual file size on <b>Disk</b>. Instead, <b>Spark</b> adds the value of configuration option <b>spark.sql.files.openCostInBytes</b>, which defaults to <b>4 MB</b>, to every single file as <b>Virtual Weight</b>.
* When an entire input file, or, chunk of data from an input file is packed into a <b>Spark Partition</b> of default size <b>128 MB</b>, which is defined by the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b>, then <b>Spark</b> computes the weight of that input file as -
    * <b>Virtual File Size</b> = <b>Actual File Size</b> + <b>openCostInBytes</b>
* <b>Handling Small Files</b>:
    * If the <b>File-Based Source</b> has thousands of tiny <b>Kilobyte Sized</b> files, then a <b>4 MB</b> of <b>file open cost</b> is added to each of those tiny <b>Kilobyte Sized</b> files.
    * This prevents <b>Spark</b> from shoving thousands of tiny files into one <b>Spark Partition</b>, which would otherwise choke the <b>Driver Memory</b> with massive <b>Metadata Loading Overhead</b>, and, would slow down the execution time of each <b>Task</b>.

# Can the Values for Spark Configuration Options "spark.sql.files.maxPartitionBytes" and "spark.sql.files.openCostInBytes" be Altered in Databricks Serverless Compute?
* <b>1</b>. <b>spark.sql.files.maxPartitionBytes</b>:
    * Yes, it is possible to alter the value of the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> on <b>Serverless Compute Cluster</b> in <b>Databricks</b>.
    * Since. the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> purely controls the size of the <b>Spark Partitions</b> when reading files from a <b>File-Based Data Source</b>. <b>Databricks</b> lists this configuration option as one of the few safe, configurable parameters to help optimize the workloads when the default value of this configuration option, i.e., <b>128 MB</b> causes too many <b>Spark Partition</b> splits, or, leads to <b>Data Skew</b> bottlenecks.
    * Because <b>Serverless Compute Cluster</b> relies entirely on <b>Spark Connect Sessions</b>, it is possible to update the value of the configuration option <b>spark.sql.files.maxPartitionBytes</b> at runtime in a <b>Notebook</b>, which can be scheduled to execute in a <b>Databricks Workflow</b> at an interval on <b>Serverless Compute Cluster</b>.
* <b>2</b>. <b>spark.sql.files.openCostInBytes</b>:
    * No, it is not possible to alter the value of the <b>Spark</b> configuration option <b>spark.sql.files.openCostInBytes</b> on <b>Serverless Compute Cluster</b> in <b>Databricks</b>.
    * If the value of the configuration option <b>spark.sql.files.openCostInBytes</b> is tried to be updated manually at runtime in a <b>Notebook</b>, which would be scheduled to execute in a <b>Databricks Workflow</b> at an interval on a <b>Serverless Compute Cluster</b>, then the <b>Databricks Workflow</b> will fail, because, it is an unsupported configuration parameter on <b>Serverless Compute Cluster</b>. 
    * The default value of the <b>Spark</b> configuration option <b>spark.sql.files.openCostInBytes</b>, i.e., <b>4 MB</b>, is locked into the <b>Serverless Compute Cluster</b>.

In [0]:
# Alter the Value of the Configuration Option "spark.sql.files.maxPartitionBytes" to 256 MB, i.e., 268435456 Bytes
spark.conf.set("spark.sql.files.maxPartitionBytes", "268435456")

# Can AQE Change the Values for Spark Configuration Options "spark.sql.files.maxPartitionBytes" and "spark.sql.files.openCostInBytes"?
* No. <b>Adaptive Query Execution</b>, i.e., <b>AQE</b> cannot change, or, override the values of the <b>Spark</b> configuration options <b>spark.sql.files.maxPartitionBytes</b>, and, <b>spark.sql.files.openCostInBytes</b>, because, these <b>Spark</b> configuration options, and, <b>AQE</b> operate at entirely different <b>Stages</b> of a <b>Spark Job</b>'s execution life cycle -
* <b>Spark Configuration Options Operate in Initial Read Stage</b>:
    * The <b>Spark</b> configuration options <b>spark.sql.files.maxPartitionBytes</b>, and, <b>spark.sql.files.openCostInBytes</b> controls the initial <b>Scan</b>, or, <b>Read Stage</b>.
    * When a <b>Spark Query</b> is executed, <b>Spark</b> reads the files from the <b>Cloud Storage Layer</b>, like - <b>S3</b>, <b>ADLS</b>, or, <b>Delta Lake</b>, and, uses the configuration option <b>spark.sql.files.maxPartitionBytes</b>, in conjunction with the other configuration option <b>spark.sql.files.openCostInBytes</b>, as a strict rule to split, or, pack those raw files into <b>Memory Based Spark Partitions</b>.
* <b>AQE Operates from First Shuffle Boundary</b>:
    * <b>AQE</b> controls the <b>Shuffle</b>, or, <b>Runtime Stage</b>, as <b>AQE</b> is designed to optimize the <b>Spark Queries</b> the moment the first <b>Shuffle Boundary</b> occurs while executing the <b>Spark Queries</b>.
    * <b>AQE</b> is only triggered after a <b>Shuffle Boundary</b> occurs, e.g., after an explicit <b>.groupBy()</b>, <b>.join()</b>, or, <b>.distinct()</b>.
    * <b>AQE</b> evaluates the actual size of the <b>Spark Partitions</b>, containing data, in <b>Memory</b>, after that <b>Shuffle Boundary</b> operation, and, adjusts things going forward.
    * <b>AQE</b> has no ability to go backward in time and re-read the initial data files to re-create the initial <b>Spark Partitions</b>.

# How Writing Phase Depends on Spark Partitions?
* The <b>Write</b> phase relies on <b>Spark Partitions</b> in the following key ways:
    * <b>Task Parallelism</b>:
        * Each <b>Spark Partition</b> is assigned to only one <b>Task</b> for execution.
        * Having more <b>Spark Partitions</b> means each <b>Spark Partition</b> would be assigned to one <b>Task</b> respectively to process, that increases the <b>Parallelism</b>, as, multiple <b>Spark Partitions</b> would be processed by multiple <b>Tasks</b> simultaneously.
        * So, the workload gets distributed faster across the <b>Executors</b> in the <b>Cluster</b>.
    * <b>File Output Generation</b>:
        * When writing to a <b>Delta Table</b>, the underlying data files get saved in <b>.parquet</b> format, where each <b>Spark Partition</b> writes to at least one distinct part <b>.parquet</b> file.
            * Too many <b>Spark Partitions</b> can lead to the generation of thousands of tiny files, i.e., can lead to the the <b>Small File Problem</b>
            * Too few <b>Spark Partitions</b> cause bottlenecks.
    * <b>Column Based Partition Pruning</b>:
        * When data is being written to a <b>Partitioned Delta Table</b> using <b>.partitionBy("partition_column_name")</b>, then <b>Spark</b> dynamically evaluates the incoming data, and, either creates a new distinct <b>Partition Directory</b> on the <b>Disk</b> based on the provided <b>Partition Column</b> value, or, selects an existing <b>Partition Directory</b> on the <b>Disk</b> that matches with the provided <b>Partition Column</b> value.
        * Inside the created, or, selected <b>Partition Directory</b> of that <b>Delta Table</b>, each <b>Spark Partition</b> would writes to at least one distinct part <b>.parquet</b> file.
    * <b>Shuffling</b>: If the incoming data requires <b>Distribution Mapping</b>, or, <b>Re-Sorting</b> before being written to a <b>Delta Table</b>, such as - creating a <b>Hash</b> to match the <b>Partitions</b> of that <b>Delta Table</b>, then the <b>Spark Partitions</b> trigger a <b>Network-Wide Shuffle</b>.

# How Spark Partitions Determine the Number of Created Underlying Parquet Files for Target Delta Tables?
* In <b>Databricks</b>, the number of <b>Spark Partitions</b> directly dictates the maximum number of underlying <b>.parquet</b> files generated per successful <b>Write</b> operation to a  <b>Target Delta Table</b>.
* Every <b>Spark Partition</b> contains data, which is to be processed by a corresponding <b>Task</b>.
<br>After processing, each <b>Task</b> writes out the processed data to a distinct underlying <b>.parquet</b> file for the <b>Target Delta Table</b>.
* When data is being written to the <b>Target Delta Table</b>, the final file count per successful <b>Write</b> operation is governed by the below mathematical relationship -
    * <b>Total Parquet Files Written</b> <= <b>Spark Partitions</b> * <b>Target Delta Table Partitions Unique to that Task</b>
* <b>Without Partitioning</b>: If the <b>Target Delta Table</b> is not <b>Partitioned</b>, or, uses <b>Liquid Clustering</b>, then each <b>Spark Partition</b> that contains data produces exactly <b>1 .parquet</b> file.
* <b>With Partitioning Using partitionBy</b>: If the <b>Target Delta Table</b> is physically <b>Partitioned</b> by a <b>Low Cardinality</b> column, e.g., <b>date</b>, or, <b>status</b>, then a single <b>Spark Partition</b>, containing data for multiple <b>dates</b>, or, <b>statuses</b>, will scatter its records to create <b>1</b>  <b>Physical .parquet</b> file for every <b>Directory</b> based on the unique values of the <b>dates</b>, or, <b>statuses</b>.

# Creation of Underlying Parquet Files for Target Delta Tables in Case of Batch Ingestion from File-Based Source
* In a <b>Batch Processing Databricks Workflow</b>, which reads files from <b>File-Based Source</b> using codes, like - <b>df.write.format("delta").save()</b>, or, <b>df.write.format("delta").saveAsTable()</b>, the number of <b>Spark Partitions</b> is highly volatile, and, depends entirely on the <b>Initial Read</b>, and, subsequent <b>Transformations</b> -
    * <b>STEP 1</b>: <b>Initial Read Partitions</b>: How many <b>Spark Partitions</b> will be created during the <b>Read</b> operation from the <b>File-Based Source</b>, is controlled by the configuration option <b>spark.sql.files.maxPartitionBytes</b>, which defaults to <b>128 MB</b>. 
    <br>That means, if an uncompressed input file of size <b>1.2 GB</b> is read by the <b>Batch Processing Databricks Workflow</b>, then <b>Spark</b> will split that file into roughly <b>10 Spark Partitions</b>, generating up to <b>10 .parquet</b> files.
    * <b>STEP 2</b>: <b>The Shuffle Effect</b>: If any <b>Wide Transformation</b>, like - <b>groupBy</b>, <b>join</b>, <b>distinct</b>, or, an explicit <b>repartition()</b> is encountered in the <b>Spark Query</b> that gets executed in the <b>Batch Processing Databricks Workflow</b>, then the data gets <b>Re-Shuffled</b> once a <b>Shuffle Boundary</b> occurs, and, a new <b>Stage</b> gets created, and, at that point, sizing each of the <b>Spark Partitions</b> shift to the value of the configuration option <b>spark.sql.shuffle.partitions</b>, which is <b>200</b>, by default, or, <b>Adaptive Query Execution</b> (<b>AQE</b>) thresholds.
    <br>So, if there are <b>200 Spark Partitions</b> immediately prior to writing to the <b>Target Delta Table</b>, then <b>Spark</b> will attempt to write up to <b>200</b> small <b>.parquet</b> files on <b>Disk</b> of <b>Target Cloud Storage</b>.
    * <b>STEP 3</b>: <b>Mitigation</b>: Right before writing to the <b>Target Delta Table</b>, if a <b>.coalesce(`<reduced-partition-number`)</b>, or, <b>.repartition(`<reduced-partition-number`)</b> is explicitly run on the data to be written to the <b>Target Delta Table</b>, then <b>Spark</b> will reduce the the maximum number of written files to the specified number, i.e., <b>`<reduced-partition-number`</b>.

# Default Behaviour of Spark Structured Streaming to Process Files in Each Micro Batch
* In <b>Spark Structured Streaming</b>, there is no restriction on how many files can be read from the <b>Directory</b> of a <b>File-Based Streaming Sources</b> in a <b>Single Micro Batch</b>.
* <b>Spark Structured Streaming</b> will continuously check the <b>Directory</b> of a <b>File-Based Streaming Source</b> for new files in every trigger interval, and, processes all the newly discovered files in a <b>Single Micro Batch</b> unless explicitly configured.

In [0]:
# Unbounded: This Stream Processes All new Files Upon Startup Without any Default Limit
streaming_df = (spark.readStream
                     .format("json") \
                     .load("/Volumes/oc_catalog/oc_schema/oc_volume/oc_input_directory")
)

query = (streaming_df.writeStream
                     .format("console")
                     .start()
)

# What Problem Arises Due to the Default Behaviour of Spark Structured Streaming to Process Files in Each Micro Batch
* If a new <b>Stream</b> is started using a new <b>Checkpoint Directory</b>, where the <b>File-Based Streaming Source</b> contains millions of historical files, say, for <b>Backfill</b> scenario, or, when the <b>Stream</b> is running in a <b>Databricks Workflow</b> that is scheduled using the trigger mode <b>availableNow = True</b>, then <b>Spark Structured Streaming</b> will naturally try to ingest all of the million of files in the very <b>First Micro Batch</b>, which usually crashes the <b>Driver Memory</b> in the <b>Cluster</b> due to massive <b>Metadata Loading Overhead</b>.

# How to Restrict Processing the Number of Files from File-Based Streaming Source in Each Micro Batch Using Spark Structured Streaming in Databricks?

## 1. maxFilesPerTrigger
* In <b>Spark Structured Streaming</b>, the configuration option <b>maxFilesPerTrigger</b> acts as a <b>Rate-Limiting</b> mechanism that sets a limitation on the number of new files to be processed in a <b>Single Micro Batch</b> from <b>File-Based Streaming Sources</b>, such as - <b>JSON</b>, <b>CSV</b>, <b>Parquet</b>, <b>Delta Lake</b>, or, <b>Auto Loader</b>, especially when the <b>Input Directory</b> of the <b>File-Based Streaming Source</b> has millions of files.
* The configuration option <b>maxFilesPerTrigger</b> must be set as a <b>.option()</b> attached to the <b>Streaming Source Interface</b>, i.e., <b>Read Stream</b> code, which is written before <b>.load()</b>.
* The configuration option <b>maxFilesPerTrigger</b> must not be set on the <b>Writer Stream Interface</b>, i.e., <b>Sink</b>, or, the <b>Spark Session</b> configurations.
* Writing the configuration option <b>.option("maxFilesPerTrigger", "200")</b> directly in the code, will be ignored by <b>Auto Loader</b>'s file-discovery mechanism.
<br>Instead, <b>.option("cloudFiles.maxFilesPerTrigger", "200")</b> must be used in <b>Auto Loader</b>.
* In <b>Spark Structured Streaming</b>, the default number of files processed per <b>Micro Batch</b> is <b>1,000</b>, which is bounded by the configuration option <b>maxFilesPerTrigger</b>.
* Similarly, in <b>Auto Loader</b>, the default number of files processed per <b>Micro Batch</b> is <b>1,000</b>, which is bounded by the configuration option <b>cloudFiles.maxFilesPerTrigger</b>.

In [0]:
# Correct Usage of "maxFilesPerTrigger" Using Spark Structured Streaming
streaming_df = (spark.readStream
                     .format("json")
                     .option("maxFilesPerTrigger", "100") # Processes At Max 100 New Files per Micro Batch
                     .schema(ddl_schema)
                     .load("/Volumes/oc_catalog/oc_schema/oc_volume/oc_input_directory")
)

In [0]:
# Correct Usage of "maxFilesPerTrigger" Using Auto Loader
# STEP 1: Define Paths
source_path = "/Volumes/oc_catalog/oc_schema/oc_volume/oc_input_directory"
checkpoint_path = "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location"
schema_path = "/Volumes/oc_catalog/oc_schema/oc_volume/schema_location"

# STEP 2: Configure Auto Loader
df_stream = (spark.readStream
                  .format("cloudFiles") # Activates Auto Loader
                  .option("cloudFiles.format", "json") # Input file format
                  .option("cloudFiles.schemaLocation", schema_path)
                  .option("cloudFiles.maxFilesPerTrigger", "10") # Processes At Max 10 New Files Per Micro Batch        
                  .load(source_path)
)

# STEP 3: Write the Stream to a Delta Table
query_batch = (df_stream.writeStream
                  .format("delta")
                  .outputMode("append")
                  .option("checkpointLocation", checkpoint_path)
                  .trigger(availableNow = True) # Only for Serverless Compute Cluster
                  .toTable("oc_catalog.oc_schema.oc_batch_table")
)

query_nrt = (df_stream.writeStream
                  .format("delta")
                  .outputMode("append")
                  .option("checkpointLocation", checkpoint_path)
                  .trigger(processingTime = "10 seconds") # Only for Classic Compute Cluster
                  .toTable("oc_catalog.oc_schema.oc_near_real_time_table")
)

## 2. maxBytesPerTrigger
* In <b>Spark Structured Streaming</b>, the configuration option <b>maxBytesPerTrigger</b> acts as a <b>Rate-Limiting</b> mechanism that sets a <b>soft limit</b> on the maximum total size of new data, in <b>Bytes</b>, that can be processed in a <b>Single Micro Batch</b> while reading files from <b>File-Based Streaming Sources</b>.
* When reading files from <b>File-Based Streaming Sources</b>, if the remaining <b>Byte</b> allocation in a <b>Micro Batch</b> is smaller than the size of the next available file on <b>Disk</b>, then since <b>Spark</b> cannot slice raw file segments mid-discovery, it will then ingest that entire file to ensure the <b>Stream</b> makes forward progress and doesn't get stuck.
    <br>Example -  If <b>maxBytesPerTrigger</b> is configured to <b>5 GB</b>, and, the <b>Input Directory</b> of the <b>File-Based Streaming Source</b> contains three files that are sized <b>2 GB</b> each, then, instead of leaving the last file behind, <b>Spark</b> will process all three files in a <b>Single Micro Batch</b>, which will proces total <b>6 GB</b> data.
* It is possible to specify values in the configuration parameter <b>maxBytesPerTrigger</b> as -
    * <b>Numbers in Bytes</b>, e.g., <b>104857600</b>
    * <b>Human-Readable Short-Hand Strings</b>, like - <b>100m</b> (<b>m</b> to denote <b>Megabytes</b>), or, <b>2g</b> (<b>g</b> to denote <b>Gigabytes</b>).
* By default, <b>Spark Structured Streaming</b>, or, <b>Auto Loader</b> does not place any <b>Byte Size</b> constraints on a <b>Micro Batch</b>.
* Writing the configuration option <b>.option("maxBytesPerTrigger", "2g")</b> directly in the code, will be ignored by <b>Auto Loader</b>'s file-discovery mechanism.
<br>Instead, <b>.option("cloudFiles.maxBytesPerTrigger", "2g")</b> must be used in <b>Auto Loader</b>.

In [0]:
# Correct Usage of "maxBytesPerTrigger" Using Spark Structured Streaming
streaming_df = (spark.readStream
                     .format("json")
                     .option("maxBytesPerTrigger", "500m") # Processes At Max 500 MB Byte Size Per Micro Batch
                     .schema(ddl_schema)
                     .load("/Volumes/oc_catalog/oc_schema/oc_volume/oc_input_directory")
)

In [0]:
# Correct Usage of "maxFilesPerTrigger" Using Auto Loader
# STEP 1: Define Paths
source_path = "/Volumes/oc_catalog/oc_schema/oc_volume/oc_input_directory"
checkpoint_path = "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location"
schema_path = "/Volumes/oc_catalog/oc_schema/oc_volume/schema_location"

# STEP 2: Configure Auto Loader
df_stream = (spark.readStream
                  .format("cloudFiles") # Activates Auto Loader
                  .option("cloudFiles.format", "json") # Input file format
                  .option("cloudFiles.schemaLocation", schema_path)
                  .option("cloudFiles.maxBytesPerTrigger", "500m") # Processes At Max 500 MB Byte Size Per Micro Batch        
                  .load(source_path)
)

# STEP 3: Write the Stream to a Delta Table
query_batch = (df_stream.writeStream
                  .format("delta")
                  .outputMode("append")
                  .option("checkpointLocation", checkpoint_path)
                  .trigger(availableNow = True) # Only for Serverless Compute Cluster
                  .toTable("oc_catalog.oc_schema.oc_batch_table")
)

query_nrt = (df_stream.writeStream
                  .format("delta")
                  .outputMode("append")
                  .option("checkpointLocation", checkpoint_path)
                  .trigger(processingTime = "10 seconds") # Only for Classic Compute Cluster
                  .toTable("oc_catalog.oc_schema.oc_near_real_time_table")
)

* For any <b>Single Micro Batch</b>, the number of files selected is -
    * <b>Files in Batch = min (Available Unprocessed Files, maxFilesPerTrigger, Files Needed to Reach maxBytesPerTrigger).
* If no limits are explicitly declared, <b>Databricks</b> reads <b>1,000</b> files, by default, from a <b>File-Based Streamig Source</b>. 

# How "maxFilesPerTrigger" Behaves in Case of Batch Trigger Mode, i.e., Using "availableNow = True"?
* When a <b>Stream</b> is created, that read files from the <b>Input Directory</b> of a <b>File-Based Streaming Source</b> using <b>Spark Structured Streaming</b>, which is executed using a <b>Databricks Workflow</b>, and, is scheduled in a <b>Batch</b> mode via the trigger mode <b>availableNow = True</b>, then <b>Spark Structured Streaming</b> splits the total available backlog files, present in the <b>Input Directory</b>, into multiple <b>Micro Batches</b>, where each <b>Micro Batch</b> strictly contains the number of files, which will be the <b>Minimum</b> between -
    * The value as specified by the configuration option <b>maxFilesPerTrigger</b>
    * The remaining available files present in the <b>File-Based Streaming Source</b>
* The <b>Batch</b> trigger mode, i.e., <b>availableNow = True</b> is allowed in <b>Serverless Compute Cluster</b>.
* Example -
    * When the <b>Streaming</b> starts, the <b>Spark</b> checks the <b>Checkpoint Directory</b> on <b>Cloud Storage</b> to record the precise state of all currently outstanding files.
    * If there are <b>5,000</b> files available in the <b>Input Directory</b>, and, the configuration option <b>maxFilesPerTrigger</b> is set to <b>2,000</b>, then <b>Spark</b> schedules <b>3 Sequential Micro Batches</b> inside that <b>Single Execution Run</b> -
        * Batch 1: 2000
        * Batch 2: 2000
        * Batch 3: 1000
    * Once the <b>5,000</b> files are consumed, the <b>Stream</b> gracefully shuts down.

# How "maxFilesPerTrigger" Behaves in Case of Near-Real Time Trigger Mode, i.e., Using 'processingTime = "`<value>` seconds"'?
* When a <b>Stream</b> is created that read files from the <b>Input Directory</b> of a <b>File-Based Streaming Source</b> using <b>Spark Structured Streaming</b>, which is executed using a <b>Databricks Workflow</b>, and, is scheduled in a <b>Near-Real Time</b> mode via the trigger mode, say, <b>`processingTime = "10 seconds"`</b>, then <b>Spark Structured Streaming</b> checks for new files every 10 seconds in the <b>Input Directory</b>, grabs any unread files it finds, and, adds those newly discovered files in the <b>Micro Batches</b>, where each <b>Micro Batch</b> strictly contains the number of files, which will be the <b>Minimum</b> between -
    * The value as specified by the configuration option <b>maxFilesPerTrigger</b>.
    * The newly discovered files present in the <b>File-Based Streaming Source</b>
* The <b>Near-Real Time</b> trigger mode, i.e., <b>`processingTime = "<value> seconds"`</b> is not allowed in <b>Serverless Compute Cluster</b>.
* Example -
    * When the <b>Streaming</b> starts, <b>Spark Structured Streaming</b> checks the <b>Input Directory</b> every <b>10 seconds</b>.
    * If only <b>50</b> new files landed in the <b>Input Directory</b> in the last <b>10 seconds</b>, then the <b>Micro Batch</b> will contain exactly <b>50</b> files, ignoring the default limitation value, imposed by the configuration option <b>maxFilesPerTrigger</b>, which is <b>1,000</b>.
    * If <b>3,000</b> new files landed in the <b>Input Directory</b> in the last <b>10 seconds</b>, then the <b>Micro Batch</b> will contain exactly <b>1,000</b> files, which is the default limitation set by the configuration option <b>maxFilesPerTrigger</b>.
    <br>The remaining <b>2,000</b> files will wait for subsequent <b>10-second Trigger Windows</b>, where after each <b>10 seconds</b> other new files can land in the <b>Input Directory</b>.

# Why Combine "maxFilesPerTrigger" with "maxBytesPerTrigger" to Restrict Processing the Number of Files from File-Based Streaming Source in Each Micro Batch Using Spark Structured Streaming / Auto Loader in Databricks?
* By default, <b>Spark Structured Streaming</b>, or, <b>Auto Loader</b> controls the size of <b>Stream</b> in each <b>Micro Batch</b> when the <b>Spark Structured Streaming</b> configuration option <b>maxFilesPerTrigger</b>, or, the <b>Auto Loader</b> configuration option <b>cloudFiles.maxFilesPerTrigger</b> is implemented, where the default value of both types of configuration options is <b>1,000</b>, or, when a value is set for the both types of configuration options, say, <b>500</b>.
* Because there is no limitation on the <b>Byte Size</b> of the data to be processed in each <b>Micro Batch</b>, say, if <b>1,000</b> files arrive in an <b>Input Directory</b> of a <b>File-Based Streaming Source</b>, where the size of each file happens to be around <b>500 MB</b>, then, <b>Spark Structured Streaming</b>, or, <b>Auto Loader</b> will attempt to pull approximately <b>500 GB</b> (<b>1000 * 500 MB = 500 GB</b> approximately) of data into a <b>Single Micro Batch</b>.
<br>This scenario frequently causes <b>Out Of Memory</b> (<b>OOM</b>) errors on <b>Driver Memory</b> in smaller <b>Clusters</b>.
* To stop causing <b>Out Of Memory</b> (<b>OOM</b>) errors on <b>Driver Memory</b>, or, to prevent massive data spikes from overwhelming the <b>Streaming</b> application, it is possible to use the configuration option <b>maxFilesPerTrigger</b> alongside the other configuration option <b>maxBytesPerTrigger</b>, which sets a limitation on the <b>Byte Size</b> of the data to be processed in each <b>Micro Batch</b>.
* When both the configuration options are defined, <b>Spark</b> is smart enough to cut off inserting the data to be processed per <b>Micro Batch</b> as soon as either default <b>1,000</b> files are queued, or, the custom <b>Byte Size</b> limit is crossed.

# Dependency of Spark Structured Streaming Configuration Options on Creation of Underlying Parquet Files for Target Delta Tables in Case of File-Based Streaming Source Using Structured Streaming
* In a <b>Stream Processing Databricks Workflow</b>, which reads files from <b>Streaming File-Based Source</b> using codes, like - <b>spark.readStream.format("parquet").save()</b>, or, <b>spark.readStream.format("delta").saveAsTable()</b>, or, using <b>Auto Loader</b>, the number of <b>Spark Partitions</b> on a <b>Micro Batch</b> level depends on -
    * <b>Micro Batch Scaling</b>: For every trigger interval, <b>Spark</b> processes a subset of newly discovered files.
    <br>The number of <b>Spark Partitions</b> for that <b>Micro Batch</b> are determined by the size of the discovered files divided by the value set to the configuration option <b>maxPartitionBytes</b>.
    * <b>The File Explosion Risk</b>: If the <b>File-Based Streaming Source</b> contains thousands of <b>Kilo Byte</b> sized tiny files, and, the trigger runs every few seconds, say, every <b>10 seconds</b>, then <b>Spark</b> will generate small number of <b>Micro Batches</b>, where each <b>Micro Batch</b> contains low number of <b>Spark Partitions</b>.
    <br>These <b>Micro Batches</b> will be executed instantly, which leads to the <b>Small File Problem</b>, i.e., thousands of tiny <b>.parquet</b> files saturating the <b>Transaction Log</b> of the <b>Target Delta Table</b>.
    * <b>Rate-Limiting Control</b>: The configuration options, like - <b>maxFilesPerTrigger</b>, or, <b>maxBytesPerTrigger</b> limit how much data enters into a <b>Single Micro Batch</b>, directly constraining the number of <b>Spark Partitions</b> per <b>Micro Batch</b> when writing the data to the <b>Target Delta Table</b>.

# How Spark Configuration Option "spark.sql.files.maxPartitionBytes" Determines the Number of Spark Partitions to Create in Initial Read Stage Using Spark Structured Streaming / AutoLoader?
* The behaviour of the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> becomes unpredictable when files are read from <b>File-Based Streaming Source</b> using <b>Spark Structured Streaming</b>, or, <b>Auto Loader</b> running in a <b>Databricks Workflow</b> that is triggered to run at a specific interval, like - <b>hourly</b>, <b>daily</b> etc, via <b>availableNow = True</b> trigger mode.
* In this specific setup, the <b>Spark</b> configuration option <b>spark.sql.files.maxPartitionBytes</b> is overridden, or, bypassed by the below higher-priority <b>Streaming</b> mechanics, because the <b>Spark Structured Streaming</b>, or, <b>Auto Loader</b> engine performs file discovery asynchronously, and, manages the data chunk limits internally before <b>Spark Standard File Readers</b> touch the files present in the <b>Input Directory</b> of the <b>File-Based Streaming Source</b> -
    * <b>The Streaming Trigger Hierarchy Overrides</b>: In this specific setup, i.e., when files are read from <b>File-Based Streaming Source</b> using <b>Spark Structured Streaming</b>, running in a <b>Databricks Workflow</b> that is triggered to run at a specific interval, like - <b>hourly</b>, <b>daily</b> etc, via <b>availableNow = True</b> trigger mode, <b>Spark</b> does not process all the data in a <b>Single Batch</b> like a standard <b>SQL</b> query. Instead <b>Spark</b> performs the below sequential operations -
        * <b>1</b>. <b>The Micro-Batch Split</b>: <b>Spark</b> splits all available files into multiple distinct <b>Micro Batches</b> before execution.
        * <b>2</b>. <b>The Controlling Properties</b>: The size, and, volume of the data of the initial <b>Spark Partitions</b> are strictly dictated by <b>Spark Structured Streaming</b> configuration options, primarily -
            * <b>maxFilesPerTrigger</b>, which default to <b>1,000</b> files.
            * <b>maxBytesPerTrigger</b>, which does not have any default value.
        * Example -
            * If the <b>File-Based Streaming Source</b> contains many file, with size combining to <b>5 GB</b>, but, the <b>Spark Structured Streaming</b> configuration option <b>maxBytesPerTrigger</b> is defined as <b>1 GB</b>, then <b>Spark</b> will create each <b>Micro Batch</b>, limiting the size to <b>1 GB</b>.
            * Then, <b>Spark</b> will only look at the first <b>Micro Batch</b> of size <b>1 GB</b>.
            * Inside each of the <b>Micro Batches</b>, the <b>Spark</b> configuration options <b>spark.sql.files.maxPartitionBytes</b>, along with <b>spark.sql.files.openCostInBytes</b> will be used to calculate how many files would be packed into each <b>Spark Partition</b> to create the corresponding <b>Tasks</b> that will process data for each of the <b>Spark Partitions</b> in that <b>Micro Batch</b>.
            * The same process will be applied to all the <b>Micro Batches</b>.
    * <b>Auto Loader, or, File-Based Streaming Source Rate Limiting Takes Precedence</b>: If <b>Auto Loader</b> is used to read files from the <b>File-Based Streaming Source</b>, then the <b>Auto Loader</b> configuration options <b>cloudFiles.maxBytesPerTrigger</b>, and, <b>cloudFiles.maxFilesPerTrigger</b> are the parameters that manage how much data to be inserted into each <b>Micro Batch</b> to process.
        * Inside each of the <b>Micro Batches</b>, the <b>Spark</b> configuration options <b>spark.sql.files.maxPartitionBytes</b>, along with <b>spark.sql.files.openCostInBytes</b> will be used to calculate how many files would be packed into each <b>Spark Partition</b> to create the corresponding <b>Tasks</b> that will process data for each of the <b>Spark Partitions</b> in that <b>Micro Batch</b>.
            * The same process will be applied to all the <b>Micro Batches</b>.

# What is "Small File Problem"?
* The <b>Small File Problem</b> occurs when continuous, <b>low-latency</b> <b>writes</b> generate a high volume of fragmented, <b>sub-optimal</b> data files.
<br><b>Small File Problem</b> mostly occurs in <b>Streaming Data</b> workloads.
* This degrades the read performance of those small files due to immense disk I/O bottlenecks and bloats the <b>metadata transaction log</b>, which makes the <b>Execution Plan</b> of the <b>Spark Structured Streaming Query</b> excruciatingly slow.

# How the "Small File Problem" Occurs in Delta Lake When Reading Files from File-Based Streaming Sources?
* <b>Spark Structured Streaming</b> ingests files from <b>File-Based Streaming Source</b> in short intervals of time in <b>Micro Batches</b> 
* If the <b>File-Based Streaming Source</b> itself contains thousands of tiny <b>Kilobyte Sized</b> files, then, the <b>Spark Structured Streaming Query</b> writes out that many number of small files to the <b>Data Lake</b>.

# How the "Small File Problem" Occurs in Delta Lake When Reading Data from Streaming Delta Tables?
* The <b>Small File Problem</b> is compounded in case of <b>Streaming Delta Tables</b>.
* Because, each successful <b>Micro Batch Write</b> to the <b>Streaming Delta Tables</b> creates a new <b>commit</b> that contains its own set of <b>Data Files</b>, writing continuously for hours, or, days to the <b>Streaming Delta Tables</b>, results in creation of millions of <b>Micro Batches</b> containing millions of fractional files rather than a few large, contiguous ones.
* Additionally, if the <b>Streaming Delta Table</b> is partitioned by <b>Low Cardinality</b> columns, e.g., <b>date</b>, or, <b>status</b>, then the <b>Spark Structured Streaming Query</b> is forced to create a <b>Directory Structure</b>, where each <b>Partition</b> contains tiny files.

# How the "Small File Problem" Occurs When Reading Messages from Messaging Queues?
* When reading from a  <b>Messaging Queue</b>, like - <b>Kafka Topic</b>, with very small numbers of messages per <b>Partition</b>, the number of small files generated per <b>Micro Batch</b> in <b>Delta Lake</b> is determined by the number of active <b>Spark Partitions</b> during the <b>Write</b> phase, multiplied by the number of distinct <b>Target Delta Table Partitions</b>.
* It does not directly correlate with the original count of the <b>Kafka Topic Partition</b>, if data <b>Shuffling</b> occurs.

# How to Solve the "Small File Problem" in Streaming Delta Tables?

## 1. Delta Lake Compaction & Optimization
* <b>Delta Lake</b> offers native mechanisms to clean up fragmented data without interrupting ongoing <b>Stream</b> by preventing small files from ever being written into the <b>Streaming Delta Table</b> -
    * <b>Auto Compaction</b>:
        * When <b>Auto Compaction</b> feature is enabled using <b>delta.autoOptimize.autoCompact</b>, it instructs <b>Delta Lake</b> to check for small files right after a <b>Micro Batch Write</b> succeeds to the <b>Streaming Delta Table</b>, and, merge those small files into larger target files, typically sized around <b>128 MB</b>.
    * <b>Optimized Writes</b>:
        * When <b>Optimized Writes</b> feature is enabled using <b>delta.autoOptimize.optimizeWrite</b>, then <b>Delta Lake</b> dynamically optimizes the sizes of the <b>Partitions</b> of the <b>Streaming Delta Table</b> based on the actual data during the <b>Micro Batch Write</b> phase, which prevents the creation of too many underlying small files in the first place.

In [0]:
# STEP 1. Define the Streaming Source Volume Path, Checkpoint Path, and, Schema Location Path
source_json_path = "/Volumes/oc_catalog/oc_schema/oc_volume/input_files/"
checkpoint_path = "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location"
schema_location_path = "/Volumes/oc_catalog/oc_schema/oc_volume/schema_location"

# STEP 2. Read Streaming JSON Files Using Auto Loader
streaming_df = (spark.readStream
                     .format("cloudFiles")
                     .option("cloudFiles.format", "json")
                     .option("cloudFiles.schemaLocation", schema_location_path)
                     .load(source_json_path)
)

# STEP 3. Write the Streaming Data to Delta Table with Auto Compaction Enabled
query = (streaming_df.writeStream
                     .format("delta")
                     .outputMode("append")
                     .option("delta.autoOptimize.autoCompact", "true") # Enables Auto Compaction During Micro Batch Write
                     .option("delta.autoOptimize.optimizeWrite", "true") # Optimizes File Sizes During Micro Batch Write
                     .trigger(availableNow = True)
                     .option("checkpointLocation", checkpoint_path)
                     .toTable("oc_catalog.oc_schema.oc_auto_compact_table")
)

## 2. OPTIMIZE Command
* g